In [11]:
pip install pandas jieba tqdm

Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import jieba
from tqdm import tqdm

In [13]:
#H3检验
#加载 NTUSD 词典
pos_path = r"C:\Users\97201\Desktop\6601 R code\sentiment_dict\ntusd_positive.txt"
neg_path = r"C:\Users\97201\Desktop\6601 R code\sentiment_dict\ntusd_negative.txt"

from opencc import OpenCC

cc = OpenCC('t2s')  # 繁体 → 简体

def load_ntusd(path):
    with open(path, encoding="cp950") as f:
        return set(
            cc.convert(line.strip())
            for line in f
            if line.strip()
        )

pos_words = load_ntusd(pos_path)
neg_words = load_ntusd(neg_path)

In [14]:
#检验是否成功
list(pos_words)[:20]

['飞弹',
 '做到了',
 '对准',
 '察觉',
 '大有机会',
 '亟待厘清',
 '精辟',
 '有先见之明',
 '尚有',
 '伦理',
 '马上',
 '未放弃',
 '很笃定',
 '开始和解',
 '慎选',
 '史上最大',
 '努力寻求',
 '踏出的一大步',
 '举行凯旋式',
 '完美的结合']

In [15]:
#情绪函数
import pandas as pd
import jieba
from tqdm import tqdm

tqdm.pandas()

def sentiment_score(text):
    if pd.isna(text):
        return 0
    words = jieba.lcut(text)
    pos = sum(1 for w in words if w in pos_words)
    neg = sum(1 for w in words if w in neg_words)
    return pos - neg


In [16]:
#新闻情绪 → 日聚合 → 导出
news = pd.read_csv(
    r"C:\Users\97201\Desktop\6601 R code\news_raw.csv",
    encoding="utf-8"
)

news["sentiment"] = news["Abstract"].progress_apply(sentiment_score)

news_daily = (
    news.groupby("date")
        .agg(
            news_sentiment_mean=("sentiment", "mean"),
            news_sentiment_sum=("sentiment", "sum"),
            news_count=("sentiment", "count")
        )
        .reset_index()
)

news_daily.to_csv(
    r"C:\Users\97201\Desktop\6601 R code\news_daily_sentiment.csv",
    index=False,
    encoding="utf-8"
)


  0%|          | 0/10793 [00:00<?, ?it/s]Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\97201\AppData\Local\Temp\jieba.cache
Loading model cost 0.417 seconds.
Prefix dict has been built successfully.
 53%|█████▎    | 5731/10793 [00:02<00:01, 2754.10it/s]


KeyboardInterrupt: 

In [9]:
weibo = pd.read_csv(
    r"C:\Users\97201\Desktop\6601 R code\weibo_raw.csv",
    encoding="utf-8"
)

weibo["sentiment"] = weibo["text_content"].progress_apply(sentiment_score)

weibo_daily = (
    weibo.groupby("date")
         .agg(
             weibo_sentiment_mean=("sentiment", "mean"),
             weibo_sentiment_sum=("sentiment", "sum"),
             weibo_count=("sentiment", "count")
         )
         .reset_index()
)

weibo_daily.to_csv(
    r"C:\Users\97201\Desktop\6601 R code\weibo_daily_sentiment.csv",
    index=False,
    encoding="utf-8"
)


100%|██████████| 3002028/3002028 [1:10:06<00:00, 713.60it/s] 


In [18]:
#H4 检验
import pandas as pd
import jieba
from collections import Counter

In [19]:
# 读取大工词典
dlut_path = r"C:\Users\97201\Desktop\6601 R code\大工情感词典（DLUT）.xlsx"
dlut = pd.read_excel(dlut_path)

# 查看列名（第一次跑建议 print 一下）
print(dlut.columns)


Index(['词语', '词性种类', '词义数', '词义序号', '情感分类', '强度', '极性', '辅助情感分类', '强度.1',
       '极性.1', 'Unnamed: 10', 'Unnamed: 11'],
      dtype='object')


In [20]:
sub_to_main = {
    # 乐
    "PA": "joy",
    "PE": "joy",

    # 好
    "PD": "good",
    "PH": "good",
    "PG": "good",
    "PB": "good",
    "PK": "good",

    # 怒
    "NA": "anger",

    # 哀
    "NB": "sadness",
    "NJ": "sadness",
    "NH": "sadness",
    "PF": "sadness",

    # 惧
    "NI": "fear",
    "NC": "fear",
    "NG": "fear",

    # 恶
    "NE": "disgust",
    "ND": "disgust",
    "NN": "disgust",
    "NK": "disgust",
    "NL": "disgust",

    # 惊
    "PC": "surprise"
}


In [21]:
emotion_labels = [
    "joy",      # 乐
    "good",     # 好
    "anger",    # 怒
    "sadness",  # 哀
    "fear",     # 惧
    "disgust",  # 恶
    "surprise"  # 惊
]

emotion_dict = {e: set() for e in emotion_labels}


In [22]:
for _, row in dlut.iterrows():
    word = str(row["词语"]).strip()
    code = str(row["情感分类"]).strip()

    if code in sub_to_main:
        main_emotion = sub_to_main[code]
        emotion_dict[main_emotion].add(word)


In [25]:
#定义函数
def compute_emotion_scores(text):
    if pd.isna(text):
        return {e: 0 for e in emotion_labels}

    words = jieba.lcut(str(text))
    total = len(words)

    if total == 0:
        return {e: 0 for e in emotion_labels}

    counter = Counter(words)

    scores = {}
    for e in emotion_labels:
        scores[e] = sum(counter[w] for w in emotion_dict[e] if w in counter) / total

    return scores



In [26]:
# 新闻端
news_emotion = news["Abstract"].apply(compute_emotion_scores)
news_emotion_df = pd.DataFrame(list(news_emotion))

news_daily = (
    pd.concat([news[["date"]], news_emotion_df], axis=1)
    .groupby("date")
    .mean()
    .reset_index()
)


In [27]:
news_daily.to_csv("news_daily_dlut_emotion.csv", index=False, encoding="utf-8-sig")

In [31]:
import pandas as pd

weibo = pd.read_csv(
    r"C:\Users\97201\Desktop\6601 R code\weibo_raw.csv"
)

weibo["date"] = pd.to_datetime(weibo["date"])

In [32]:
weibo.head()
weibo.columns


Index(['date', 'text_content'], dtype='object')

In [33]:
from tqdm import tqdm
tqdm.pandas()


In [35]:
def compute_emotion_scores(text):
    if pd.isna(text):
        return {e: 0 for e in emotion_labels}

    words = jieba.lcut(str(text))
    total = len(words)

    if total == 0:
        return {e: 0 for e in emotion_labels}

    counter = Counter(words)

    scores = {}
    for e in emotion_labels:
        scores[e] = sum(counter[w] for w in emotion_dict[e] if w in counter) / total

    return scores


In [37]:
print("Start computing Weibo emotion scores...")

weibo_emotion = weibo["text_content"].progress_apply(compute_emotion_scores)

weibo_emotion_df = pd.DataFrame(list(weibo_emotion))



Start computing Weibo emotion scores...


100%|██████████| 3002028/3002028 [2:44:39<00:00, 303.88it/s]  


In [41]:
weibo_daily = (
    pd.concat([weibo[["date"]], weibo_emotion_df], axis=1)
    .groupby("date")
    .mean()
    .reset_index()
)

In [42]:
weibo_daily.to_csv(
    r"C:\Users\97201\Desktop\6601 R code\weibo_daily_emotion_dlutt.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Weibo daily emotion file exported successfully!")


Weibo daily emotion file exported successfully!
